# Training — Major vs Minor Chord Classifier (Colab)

This notebook trains the whole model zoo (RF, SVM, KNN, LR, MLP, CNN, LSTM, GRU)
and saves each one into `models/<name>/`. All the real work lives in `src/`, so
this notebook only:

1. gets the code and the datasets, then
2. calls `src.train.main(...)`.

**Recommended:** Runtime → Change runtime type → GPU (the CNN/RNN train much
faster). The classical models don't need a GPU.


## 1. Get the code

In [ ]:
# The Colab kernel runs on a remote machine (cwd = /content), so it can't see the
# project files that sit next to this notebook on your computer. Put a copy of the
# project on the server, then make it the working dir + import root
# (src.config resolves models/ and data/ relative to it).
#
# Option A (default): clone from GitHub -- commit + push src/ to BRANCH first
#                     (the kernel only ever sees what is on GitHub).
# Option B: upload the ChordClassifier folder to Google Drive and set USE_DRIVE = True.
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Stelllarce/ChordClassifier.git'
BRANCH = 'colab-training'
USE_DRIVE = False
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/ChordClassifier'
CLONE_DIR = Path('/content/ChordClassifier')

# Every module the notebook imports -- a checkout missing any of them is stale.
REQUIRED = ['__init__.py', 'config.py', 'features.py', 'dataset.py',
            'models.py', 'persistence.py', 'train.py']

def missing(d):
    return [f for f in REQUIRED if not (Path(d) / 'src' / f).exists()]

def git(*args, cwd=None):
    subprocess.run(['git', *args], cwd=cwd, check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
elif not missing(Path.cwd()):
    PROJECT_DIR = Path.cwd()                 # local Jupyter kernel inside the project
elif not missing(Path.cwd().parent):
    PROJECT_DIR = Path.cwd().parent          # local Jupyter kernel inside notebooks/
else:
    PROJECT_DIR = CLONE_DIR
    if (CLONE_DIR / '.git').exists():
        # Re-running: pull the latest pushed code instead of cloning into a non-empty dir.
        git('fetch', '--depth', '1', 'origin', BRANCH, cwd=CLONE_DIR)
        git('checkout', '-B', BRANCH, 'FETCH_HEAD', cwd=CLONE_DIR)
        git('reset', '--hard', 'FETCH_HEAD', cwd=CLONE_DIR)
    else:
        shutil.rmtree(CLONE_DIR, ignore_errors=True)
        git('clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(CLONE_DIR))

PROJECT_DIR = PROJECT_DIR.resolve()
gaps = missing(PROJECT_DIR)
assert not gaps, (f'{PROJECT_DIR} is missing src/{gaps}. The Colab kernel only sees what is '
                  f'pushed to GitHub -- commit + push src/ to branch {BRANCH!r}, then re-run this cell.')

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
# Forget any `src` imported earlier in this session (old checkout / other path),
# so `import src.train` below loads the code that's on disk now.
for mod in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[mod]
print('Project root:', PROJECT_DIR)

!pip -q install librosa kagglehub

## 2. Download the datasets

In [ ]:
import os
from pathlib import Path
import kagglehub
from src import config

shords_path = kagglehub.dataset_download("mehanat96/major-vs-minor-guitar-chords")
audio_files_path = kagglehub.dataset_download("deepcontractor/musical-instrument-chord-classification")

data_dir = config.DATA_DIR; data_dir.mkdir(exist_ok=True)

def class_root(src):
    """kagglehub returns .../<version>/ with the dataset folder nested inside;
    descend through single-folder wrappers to the dir whose subdirs are the labels."""
    src = Path(src)
    while True:
        subdirs = [d for d in src.iterdir() if d.is_dir() and not d.name.startswith('.')]
        has_audio = any(f.suffix.lower() in config.AUDIO_EXTENSIONS for f in src.iterdir() if f.is_file())
        if len(subdirs) != 1 or has_audio:
            return src
        src = subdirs[0]

def link(src, dest):
    dest = Path(dest)
    if dest.is_symlink():
        dest.unlink()  # re-point links left over from an earlier run
    if not dest.exists():
        os.symlink(class_root(src), dest)

link(shords_path, data_dir / 'shords_dataset')
link(audio_files_path, data_dir / 'Audio_Files')
print('shords:', sorted(p.name for p in (data_dir / 'shords_dataset').iterdir()))
print('audio :', sorted(p.name for p in (data_dir / 'Audio_Files').iterdir()))

## 3. Train everything

`epochs` controls the deep models (classical models ignore it). 80 is a good
default on a GPU. Use `train.main(quick=True)` for a fast sanity check, or
`models_to_train=['cnn','lstm','gru']` to train a subset.


In [ ]:
import src.train as train
results = train.main(epochs=80)


## 4. Compare the models

Reads the freshly-written `config.json` from each model folder and plots test F1.


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from src import persistence

cfgs = persistence.discover_models()
df = (pd.DataFrame(cfgs).T[['model_type', 'feature_type', 'cv_f1', 'test_f1', 'test_acc']]
      .sort_values('test_f1', ascending=False))
display(df)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['coral' if t == 'keras' else 'steelblue' for t in df['model_type']]
ax.barh(df.index[::-1], df['test_f1'][::-1], color=colors[::-1])
ax.set_xlabel('Test F1 (weighted)'); ax.set_title('Model comparison')
ax.set_xlim(left=max(0, df['test_f1'].min() - 0.1))
plt.tight_layout(); plt.show()


## 5. LSTM vs GRU

The assignment asks to compare the two recurrent architectures. Both are trained
above; here's their head-to-head.


In [ ]:
rnns = {k: v for k, v in persistence.discover_models().items() if k in ('lstm', 'gru')}
for name, c in rnns.items():
    print(f"{name.upper():4s}  test F1 = {c['test_f1']}  test acc = {c['test_acc']}")


## 6. Download the trained models

Zip the `models/` folder and download it, then commit it back into the repo so the
Streamlit app uses the new models.


In [ ]:
# !zip -r models.zip models
# from google.colab import files; files.download('models.zip')
